In [13]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    ExtraTreesRegressor
)


In [14]:
dataset = pd.read_csv("insurance_pre.csv")

In [15]:
print("Dataset Shape:", dataset.shape)
print(dataset.info())
print(dataset.head())

Dataset Shape: (1338, 6)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   charges   1338 non-null   float64
dtypes: float64(2), int64(2), object(2)
memory usage: 62.8+ KB
None
   age     sex     bmi  children smoker      charges
0   19  female  27.900         0    yes  16884.92400
1   18    male  33.770         1     no   1725.55230
2   28    male  33.000         3     no   4449.46200
3   33    male  22.705         0     no  21984.47061
4   32    male  28.880         0     no   3866.85520


In [16]:
print("Missing Values:")
print(dataset.isnull().sum())

print("Duplicate Rows:", dataset.duplicated().sum())


Missing Values:
age         0
sex         0
bmi         0
children    0
smoker      0
charges     0
dtype: int64
Duplicate Rows: 1


In [17]:
dataset = dataset.drop_duplicates()

In [18]:
X = dataset.drop("charges", axis=1)
y = dataset["charges"]


In [19]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [20]:
preprocessor = ColumnTransformer(
    transformers=[
        ("categorical",
         OneHotEncoder(handle_unknown="ignore"),
         ["sex", "smoker"])
    ],
    remainder="passthrough"
)

In [21]:
models = {
    "Linear Regression": LinearRegression(),

    "Decision Tree": DecisionTreeRegressor(
        random_state=42
    ),

    "Random Forest": RandomForestRegressor(
        n_estimators=200, random_state=42
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        random_state=42
    ),

    "Extra Trees": ExtraTreesRegressor(
        n_estimators=200, random_state=42
    )}

In [22]:
results = []
trained_models = {}

for name, model in models.items():

    pipeline = Pipeline([
        ("preprocessing", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)

    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)

    results.append({
        "Model": name,
        "R2 Score": round(r2, 4),
        "MAE": round(mae, 2)
    })

    trained_models[name] = pipeline

In [23]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(
    by="R2 Score", ascending=False
)

print(results_df)

               Model  R2 Score      MAE
3  Gradient Boosting    0.8999  2525.85
2      Random Forest    0.8747  2726.31
4        Extra Trees    0.8456  2841.37
0  Linear Regression    0.8046  4198.11
1      Decision Tree    0.7831  3091.12


In [24]:
best_model_name = results_df.iloc[0]["Model"]
final_model = trained_models[best_model_name]

print("Best Model:", best_model_name)


Best Model: Gradient Boosting


In [25]:

new_customer = pd.DataFrame({
    "age": [30],
    "sex": ["female"],
    "bmi": [25.5],
    "children": [1],
    "smoker": ["no"]
})

prediction = final_model.predict(new_customer)

print("Predicted Insurance Charges:", prediction[0])


Predicted Insurance Charges: 5387.250856777599
